In [1]:
##imports

import pandas as pd
import pickle
from pathlib import Path
from typing import Any, Iterable, Optional, Sequence, Set, Tuple, Union
import logging

## Set logging for visualization
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

In [2]:
##define constants

COMMON_ID_FIELDS: Tuple[str, ...] = (
    "names",
    "smiles", "SMILES", "canonical_smiles", "cano_smiles",
    "mol_id", "molid", "molecule_id", "id",
    "inchi", "InChI", "inchi_key", "InChIKey",
    "cid", "index",
)

#COMMON_ID_FIELDS

In [3]:
## load 43K list of molecules


def load_whitelist(paths: Sequence[Union[str, Path]], column: Optional[str]) -> Set[str]:
    ids: Set[str] = set()
    for p in paths:
        p = Path(p)
        if p.suffix.lower() in {".csv", ".tsv"}:
            sep = "," if p.suffix.lower() == ".csv" else "\t"
            df = pd.read_csv(p, sep=sep)
            col = column
            if col is None:
                for cand in COMMON_ID_FIELDS:
                    if cand in df.columns:
                        col = cand
                        break
            vals = df[col].astype(str).str.strip()
            ids.update(v for v in vals if v)
        else:
            with p.open("r", encoding="utf-8") as fh:
                for line in fh:
                    val = line.strip()
                    if val:
                        ids.add(val)
    return ids

whitelist_ids = load_whitelist(["/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/my_43K.txt"], None)
print(len(whitelist_ids), list(whitelist_ids)[:10])

43488 ['gdb_109076', 'gdb_11365', 'gdb_81591', 'gdb_87674', 'gdb_121353', 'gdb_25495', 'gdb_94175', 'gdb_6075', 'gdb_15726', 'gdb_51008']


In [4]:
raw_ids = load_whitelist(["/home/suba/Documents/GitHub/qtaim_embed_private/data_suba//my_43K.txt"], None)
print("Raw whitelist size:", len(raw_ids))


Raw whitelist size: 43488


In [5]:
## Load pickle datasets


def load_pickle_any(path: Union[str, Path]) -> Any:
    p = Path(path)
    try:
        return pd.read_pickle(p)
    except Exception:
        with p.open("rb") as fh:
            return pickle.load(fh)


train_raw = load_pickle_any("/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/train_qm9_qtaim_1205_labelled_corrected.pkl")
test_raw = load_pickle_any("/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/test_qm9_qtaim_1205_labelled_corrected.pkl")
type(train_raw)

pandas.core.frame.DataFrame

In [6]:
##Convert pickle content to DataFrame

def detect_id_field(columns: Iterable[str]) -> Optional[str]:
    """Auto-detect ID column from common candidates."""
    for cand in COMMON_ID_FIELDS:
        if cand in columns:
            return cand
    return None

def to_dataframe(data: Any) -> tuple[pd.DataFrame, str, str]:
    """Convert pickle content into a DataFrame, detect id_field, return (df, kind, id_field)."""
    if isinstance(data, pd.DataFrame):
        id_field = detect_id_field(data.columns)
        return data.copy(), "dataframe", id_field or ""
    if isinstance(data, list) and data and all(isinstance(x, dict) for x in data):
        df = pd.DataFrame(data)
        id_field = detect_id_field(df.columns)
        return df, "list_of_dicts", id_field or ""
    raise TypeError("Unsupported pickle structure. Must be DataFrame or list[dict].")


train_df, train_kind, train_id_field = to_dataframe(train_raw)
display(train_df.head())
print("Kind:", train_kind)
print("Detected ID field:", train_id_field)

,molecule,molecule_graph,ids,names,bonds,extra_feat_atom_Lagrangian_K,extra_feat_atom_Hamiltonian_K,extra_feat_atom_e_density,extra_feat_atom_lap_e_density,extra_feat_atom_e_loc_func,...,mu,A,B,C,r2,homo,lumo,gap,zpve,corrected_E
0,"[[-0.7092 0.8845 0.7051] O, [ 0.1492 0.9088...",Molecule Graph\nMolecule: \nFull Formula (H7 C...,120226,gdb_48609.xyz,"[[(1, 9), (0, 1), (5, 12), (1, 1), (1, 5), (1,...","[50.2037056, 5.541977253, 5.541977253, 5.73584...","[291280.9284, 64912.4472, 64912.4472, 64801.49...","[-291280.9284, -64912.4472, -64912.4472, -6480...","[-1164922.899, -259627.6209, -259627.6209, -25...","[0.9999978541, 0.9999994606, 0.9999994606, 0.9...",...,1.6230,2.71484,1.36611,1.06442,1125.7433,-0.2322,-0.0437,0.1885,0.122906,0.087413
2,"[[-0.0268 1.5235 0.0926] C, [-0.0555 0.0203...",Molecule Graph\nMolecule: \nFull Formula (H10 ...,70434,gdb_78284.xyz,"[[(5, 15), (5, 16), (5, 5), (4, 14), (5, 7), (...","[5.797971584, 5.754509552, 5.754509552, 5.8293...","[64650.0784, 64620.95207, 64620.95207, 64661.5...","[-64650.0784, -64620.95207, -64620.95207, -646...","[-258577.1217, -258460.7902, -258460.7902, -25...","[0.9999994026, 0.9999994107, 0.9999994107, 0.9...",...,2.5767,2.26905,1.71362,1.58669,984.5705,-0.2362,0.0632,0.2994,0.158274,0.031282
3,"[[0.0692 1.3948 0.1609] N, [-0.0052 0.0321 0...",Molecule Graph\nMolecule: \nFull Formula (H3 C...,40274,gdb_21533.xyz,"[[(3, 3), (2, 3), (3, 4), (2, 2), (2, 7), (4, ...","[19.22427617, 5.99512379, 5.99512379, 18.10095...","[144471.3067, 64695.26636, 64695.26636, 145612...","[-144471.3067, -64695.26636, -64695.26636, -14...","[-577808.3295, -258757.085, -258757.085, -5823...","[0.9999987007, 0.9999993626, 0.9999993626, 0.9...",...,7.2250,3.70874,1.99807,1.30024,834.7801,-0.2461,-0.0181,0.2280,0.074412,0.008412
4,"[[ 1.1000e-03 1.4643e+00 -1.6200e-02] C, [-0....",Molecule Graph\nMolecule: \nFull Formula (H11 ...,112360,gdb_95862.xyz,"[[(7, 8), (4, 16), (7, 7), (4, 7), (3, 14), (3...","[5.635737881, 18.96906908, 5.675920779, 5.6759...","[64625.44316, 144747.8903, 64643.3894, 64643.3...","[-64625.44316, -144747.8903, -64643.3894, -646...","[-258479.2297, -578915.6848, -258550.8539, -25...","[0.999999435, 0.9999987419, 0.9999994274, 0.99...",...,4.5187,2.91265,1.16741,0.87952,1321.4577,-0.2280,0.0269,0.2549,0.173271,0.002322
5,"[[ 0.0173 1.5474 -0.3641] C, [ 0.0382 0.0247...",Molecule Graph\nMolecule: \nFull Formula (H12 ...,76792,gdb_57347.xyz,"[[(3, 16), (7, 20), (2, 13), (7, 8), (3, 20), ...","[5.558948145, 5.698990077, 5.70187204, 5.69899...","[64645.19287, 64630.97815, 64631.19287, 64630....","[-64645.19287, -64630.97815, -64631.19287, -64...","[-258558.5357, -258501.1166, -258501.964, -258...","[0.9999994507, 0.9999994224, 0.9999994218, 0.9...",...,2.1097,1.99516,1.48858,1.05357,1237.2030,-0.2531,-0.0200,0.2331,0.181281,-0.025622


Kind: dataframe
Detected ID field: names


In [7]:
##Utilities: filter DataFrame, restore type


import re


def filter_df_to_ids(df: pd.DataFrame, ids: Set[str], id_field: str) -> pd.DataFrame:
    # normalize dataframe column by stripping .xyz
    df_ids = df[id_field].astype(str).str.replace(".xyz", "", regex=False)
    # normalize whitelist too (just in case some entries had .xyz)
    ids_normalized = {x.replace(".xyz", "") for x in ids}
    mask = df_ids.isin(ids_normalized)
    return df.loc[mask].copy()

def restore_to_original_type(df: pd.DataFrame, kind: str) -> Any:
    if kind == "dataframe":
        return df
    if kind == "list_of_dicts":
        return df.to_dict(orient="records")
    raise ValueError(f"Unknown kind: {kind}")

##Summarize before/after filtering

def summarize_split(name: str, df: pd.DataFrame, id_field: str, ids: Set[str]) -> None:
    df_ids = df[id_field].astype(str).str.replace(".xyz", "", regex=False)
    ids_normalized = {x.replace(".xyz", "") for x in ids}
    n_rows = len(df)
    n_unique = df_ids.nunique()
    cov = (n_unique / max(1, len(ids_normalized))) * 100.0
    print(f"{name}: rows={n_rows}, unique_ids={n_unique}, whitelist_size={len(ids_normalized)}, coverage={cov:.2f}%")


In [8]:
train_df, train_kind, train_id_field = to_dataframe(train_raw)
test_df, test_kind, test_id_field = to_dataframe(test_raw)

In [9]:
id_field = "names"  # or train_id_field
# keep whitelist_ids untouched (no .xyz added)

In [10]:
summarize_split("train(before)", train_df, id_field, whitelist_ids)
summarize_split("test(before)", test_df, id_field, whitelist_ids)

train(before): rows=120463, unique_ids=120463, whitelist_size=43488, coverage=277.00%
test(before): rows=13385, unique_ids=13385, whitelist_size=43488, coverage=30.78%


In [11]:
train_filtered = filter_df_to_ids(train_df, whitelist_ids, id_field)
test_filtered  = filter_df_to_ids(test_df, whitelist_ids, id_field)

In [12]:
summarize_split("train(after)", train_filtered, id_field, whitelist_ids)
summarize_split("test(after)", test_filtered, id_field, whitelist_ids)

train(after): rows=39121, unique_ids=39121, whitelist_size=43488, coverage=89.96%
test(after): rows=4355, unique_ids=4355, whitelist_size=43488, coverage=10.01%


In [13]:
display(train_filtered.head())

,molecule,molecule_graph,ids,names,bonds,extra_feat_atom_Lagrangian_K,extra_feat_atom_Hamiltonian_K,extra_feat_atom_e_density,extra_feat_atom_lap_e_density,extra_feat_atom_e_loc_func,...,mu,A,B,C,r2,homo,lumo,gap,zpve,corrected_E
2,"[[-0.0268 1.5235 0.0926] C, [-0.0555 0.0203...",Molecule Graph\nMolecule: \nFull Formula (H10 ...,70434,gdb_78284.xyz,"[[(5, 15), (5, 16), (5, 5), (4, 14), (5, 7), (...","[5.797971584, 5.754509552, 5.754509552, 5.8293...","[64650.0784, 64620.95207, 64620.95207, 64661.5...","[-64650.0784, -64620.95207, -64620.95207, -646...","[-258577.1217, -258460.7902, -258460.7902, -25...","[0.9999994026, 0.9999994107, 0.9999994107, 0.9...",...,2.5767,2.26905,1.71362,1.58669,984.5705,-0.2362,0.0632,0.2994,0.158274,0.031282
3,"[[0.0692 1.3948 0.1609] N, [-0.0052 0.0321 0...",Molecule Graph\nMolecule: \nFull Formula (H3 C...,40274,gdb_21533.xyz,"[[(3, 3), (2, 3), (3, 4), (2, 2), (2, 7), (4, ...","[19.22427617, 5.99512379, 5.99512379, 18.10095...","[144471.3067, 64695.26636, 64695.26636, 145612...","[-144471.3067, -64695.26636, -64695.26636, -14...","[-577808.3295, -258757.085, -258757.085, -5823...","[0.9999987007, 0.9999993626, 0.9999993626, 0.9...",...,7.2250,3.70874,1.99807,1.30024,834.7801,-0.2461,-0.0181,0.2280,0.074412,0.008412
4,"[[ 1.1000e-03 1.4643e+00 -1.6200e-02] C, [-0....",Molecule Graph\nMolecule: \nFull Formula (H11 ...,112360,gdb_95862.xyz,"[[(7, 8), (4, 16), (7, 7), (4, 7), (3, 14), (3...","[5.635737881, 18.96906908, 5.675920779, 5.6759...","[64625.44316, 144747.8903, 64643.3894, 64643.3...","[-64625.44316, -144747.8903, -64643.3894, -646...","[-258479.2297, -578915.6848, -258550.8539, -25...","[0.999999435, 0.9999987419, 0.9999994274, 0.99...",...,4.5187,2.91265,1.16741,0.87952,1321.4577,-0.2280,0.0269,0.2549,0.173271,0.002322
10,"[[-0.1132 1.3581 -0.0958] O, [ 0.0456 -0.0472...",Molecule Graph\nMolecule: \nFull Formula (H8 C...,114053,gdb_23567.xyz,"[[(6, 14), (6, 13), (6, 7), (6, 6), (7, 15), (...","[50.33421718, 17.80863605, 5.709015982, 5.7728...","[291121.8674, 145724.8276, 64665.71701, 64657....","[-291121.8674, -145724.8276, -64665.71701, -64...","[-1164286.133, -582828.0758, -258640.032, -258...","[0.9999978395, 0.9999989125, 0.9999994212, 0.9...",...,2.6176,3.10147,1.49838,1.33054,1001.3072,-0.2328,0.0258,0.2586,0.137673,0.099708
14,"[[-0.0039 1.5517 0.0146] C, [ 0.0063 0.0447...",Molecule Graph\nMolecule: \nFull Formula (H9 C...,108231,gdb_68168.xyz,"[[(6, 15), (5, 6), (5, 7), (5, 5), (5, 14), (7...","[5.597979322, 5.619102815, 5.660110186, 5.6086...","[64751.2942, 64688.20238, 64723.38508, 64846.2...","[-64751.2942, -64688.20238, -64723.38508, -648...","[-258982.7849, -258730.3331, -258870.8999, -25...","[0.9999994456, 0.9999994399, 0.9999994325, 0.9...",...,3.6853,3.60560,1.45772,1.24488,1037.6115,-0.2514,0.0179,0.2692,0.148646,-0.008173


In [14]:
#save results

def save_filtered(df: pd.DataFrame, out_path: str, as_csv: bool = False):
    p = Path(out_path)
    df.to_pickle(p)
    if as_csv:
        df.to_csv(p.with_suffix(".csv"), index=False)
    print(f"Saved: {p}")

save_filtered(train_filtered, "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/train_my43k.pkl", as_csv=True)
save_filtered(test_filtered, "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/test_my43k.pkl", as_csv=True)

Saved: /home/suba/Documents/GitHub/qtaim_embed_private/data_suba/train_my43k.pkl
Saved: /home/suba/Documents/GitHub/qtaim_embed_private/data_suba/test_my43k.pkl


In [15]:
import pandas as pd

train_csv_df = pd.read_csv("/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/train_my43k.csv")
test_csv_df = pd.read_csv("/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/test_my43k.csv")

display(train_csv_df.head())
#display(test_csv_df.head())

,molecule,molecule_graph,ids,names,bonds,extra_feat_atom_Lagrangian_K,extra_feat_atom_Hamiltonian_K,extra_feat_atom_e_density,extra_feat_atom_lap_e_density,extra_feat_atom_e_loc_func,...,mu,A,B,C,r2,homo,lumo,gap,zpve,corrected_E
0,Full Formula (H10 C7 O2)\nReduced Formula: H10...,Molecule Graph\nMolecule: \nFull Formula (H10 ...,70434,gdb_78284.xyz,"[[(5, 15), (5, 16), (5, 5), (4, 14), (5, 7), (...",[5.79797158e+00 5.75450955e+00 5.75450955e+00 ...,[6.46500784e+04 6.46209521e+04 6.46209521e+04 ...,[-6.46500784e+04 -6.46209521e+04 -6.46209521e+...,[-2.58577122e+05 -2.58460790e+05 -2.58460790e+...,[0.9999994 0.99999941 0.99999941 0.9999994 0...,...,2.5767,2.26905,1.71362,1.58669,984.5705,-0.2362,0.0632,0.2994,0.158274,0.031282
1,Full Formula (H3 C3 N5)\nReduced Formula: H3C3...,Molecule Graph\nMolecule: \nFull Formula (H3 C...,40274,gdb_21533.xyz,"[[(3, 3), (2, 3), (3, 4), (2, 2), (2, 7), (4, ...",[1.92242762e+01 5.99512379e+00 5.99512379e+00 ...,[1.44471307e+05 6.46952664e+04 6.46952664e+04 ...,[-1.44471307e+05 -6.46952664e+04 -6.46952664e+...,[-5.77808329e+05 -2.58757085e+05 -2.58757085e+...,[0.9999987 0.99999936 0.99999936 0.99999887 0...,...,7.2250,3.70874,1.99807,1.30024,834.7801,-0.2461,-0.0181,0.2280,0.074412,0.008412
2,Full Formula (H11 C6 N3)\nReduced Formula: H11...,Molecule Graph\nMolecule: \nFull Formula (H11 ...,112360,gdb_95862.xyz,"[[(7, 8), (4, 16), (7, 7), (4, 7), (3, 14), (3...",[5.63573788e+00 1.89690691e+01 5.67592078e+00 ...,[6.46254432e+04 1.44747890e+05 6.46433894e+04 ...,[-6.46254432e+04 -1.44747890e+05 -6.46433894e+...,[-2.58479230e+05 -5.78915685e+05 -2.58550854e+...,[0.99999943 0.99999874 0.99999943 0.99999943 0...,...,4.5187,2.91265,1.16741,0.87952,1321.4577,-0.2280,0.0269,0.2549,0.173271,0.002322
3,Full Formula (H8 C6 N2 O1)\nReduced Formula: H...,Molecule Graph\nMolecule: \nFull Formula (H8 C...,114053,gdb_23567.xyz,"[[(6, 14), (6, 13), (6, 7), (6, 6), (7, 15), (...",[5.03342172e+01 1.78086361e+01 5.70901598e+00 ...,[2.91121867e+05 1.45724828e+05 6.46657170e+04 ...,[-2.91121867e+05 -1.45724828e+05 -6.46657170e+...,[-1.16428613e+06 -5.82828076e+05 -2.58640032e+...,[0.99999784 0.99999891 0.99999942 0.99999941 0...,...,2.6176,3.10147,1.49838,1.33054,1001.3072,-0.2328,0.0258,0.2586,0.137673,0.099708
4,Full Formula (H9 C6 N1 O2)\nReduced Formula: H...,Molecule Graph\nMolecule: \nFull Formula (H9 C...,108231,gdb_68168.xyz,"[[(6, 15), (5, 6), (5, 7), (5, 5), (5, 14), (7...",[5.59797932e+00 5.61910281e+00 5.66011019e+00 ...,[6.47512942e+04 6.46882024e+04 6.47233851e+04 ...,[-6.47512942e+04 -6.46882024e+04 -6.47233851e+...,[-2.58982785e+05 -2.58730333e+05 -2.58870900e+...,[0.99999945 0.99999944 0.99999943 0.99999945 0...,...,3.6853,3.60560,1.45772,1.24488,1037.6115,-0.2514,0.0179,0.2692,0.148646,-0.008173


In [16]:
# Load whitelist file
with open("/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/my_43K.txt") as f:
    whitelist_ids = {line.strip() for line in f if line.strip()}
    
    
# Normalize both whitelist and dataset IDs to be consistent (remove .xyz)
wl_ids = {x.replace(".xyz", "") for x in whitelist_ids}
train_ids = set(train_df["names"].astype(str).str.replace(".xyz", "", regex=False))
test_ids = set(test_df["names"].astype(str).str.replace(".xyz", "", regex=False))

# Combine train+test IDs
all_ids = train_ids | test_ids

# Find missing IDs
missing_ids = wl_ids - all_ids
print("Missing IDs count:", len(missing_ids))
print("Some missing IDs:", list(missing_ids)[:20])

missing_id = list(missing_ids)[0]

print("Missing whitelist ID:", missing_id)
print("Does it exist in train?", any(missing_id in x for x in train_df["names"].astype(str)))
print("Does it exist in test?", any(missing_id in x for x in test_df["names"].astype(str)))

print("Whitelist size:", len(wl_ids))
print("Intersection size:", len(wl_ids & all_ids))
print("Missing size:", len(missing_ids))


import pandas as pd

pd.DataFrame({"missing_ids": sorted(missing_ids)}).to_csv("/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/missing_ids.csv", index=False)
print("Saved missing IDs to missing_ids.csv")


Missing IDs count: 12
Some missing IDs: ['gdb_40564', 'gdb_99280', 'gdb_58107', 'gdb_13776', 'gdb_7384', 'gdb_131115', 'gdb_126319', 'gdb_114983', 'gdb_41724', 'gdb_60954', 'gdb_43254', 'gdb_27992']
Missing whitelist ID: gdb_40564
Does it exist in train? False
Does it exist in test? False
Whitelist size: 43488
Intersection size: 43476
Missing size: 12
Saved missing IDs to missing_ids.csv


In [19]:
# Combine filtered train + test
combined_df = pd.concat([train_df, test_df], ignore_index=True)

# Write combined PKL
out_combined_pkl = Path("/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/combined_train_test_43k.pkl")
combined_df.to_pickle(out_combined_pkl)
logging.info(f"Wrote combined: {out_combined_pkl}")

# Write combined CSV
out_combined_csv = out_combined_pkl.with_suffix("")  # strip .pkl
out_combined_csv = out_combined_csv.with_name(out_combined_csv.name + ".csv")
combined_df.to_csv(out_combined_csv, index=False)
logging.info(f"Wrote combined: {out_combined_csv}")

[INFO] Wrote combined: /home/suba/Documents/GitHub/qtaim_embed_private/data_suba/combined_train_test_43k.pkl
[INFO] Wrote combined: /home/suba/Documents/GitHub/qtaim_embed_private/data_suba/combined_train_test_43k.csv
